# Σύγκριση F1 ανά κλάση

Σύγκριση του MobileNetV1 με παγωμένη βάση και του
MobileNetV1 με κλειδωμένα επίπεδα Batch Normalization.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


REPO_DIR = Path("/kaggle/working/thesis")

FROZEN_ZIP = (
    REPO_DIR
    / "results"
    / "frozen_models_clean_split_test_evaluation_results.zip"
)

BN_LOCKED_ZIP = (
    REPO_DIR
    / "results"
    / "mobilenet"
    / "mobilenet_clean_split_bn_locked_results.zip"
)


# Διαβάζουμε τα δύο classification reports.
with ZipFile(FROZEN_ZIP) as zip_file:
    frozen_report = pd.read_csv(
        zip_file.open(
            "mobilenetv1/classification_report.csv"
        ),
        index_col=0
    )

with ZipFile(BN_LOCKED_ZIP) as zip_file:
    bn_locked_report = pd.read_csv(
        zip_file.open("classification_report.csv"),
        index_col=0
    )

In [ ]:
# Οι πρώτες 35 γραμμές είναι οι κλάσεις.
class_names = frozen_report.index[:35]


# Κρατάμε το F1 και το support κάθε κλάσης.
heatmap_data = pd.DataFrame({
    "Frozen": frozen_report.loc[
        class_names, "f1-score"
    ],

    "BN-locked": bn_locked_report.loc[
        class_names, "f1-score"
    ],

    "support": frozen_report.loc[
        class_names, "support"
    ].astype(int)
})


# Εμφανίζουμε πρώτες τις κλάσεις
# με τις λιγότερες εικόνες στο test.
heatmap_data = heatmap_data.sort_values("support")


class_labels = [
    f"{class_name} (n={row['support']})"
    for class_name, row in heatmap_data.iterrows()
]


plt.figure(figsize=(8, 14))

sns.heatmap(
    heatmap_data[["Frozen", "BN-locked"]],
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    yticklabels=class_labels,
    cbar_kws={"label": "F1-score"}
)

plt.title("MobileNetV1 — F1-score ανά κλάση")
plt.xlabel("Μοντέλο")
plt.ylabel("Κλάση και αριθμός εικόνων στο test")

plt.tight_layout()

plt.savefig(
    "/kaggle/working/mobilenet_f1_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()